# 01 - Baseline: standard GPT, fixed batch size

Model: 12 layers, d_model 384, 6 heads, context 512, GPT-2 vocab (tied embeddings)
-> **21.3M non-embedding parameters** (40.6M including the tied 50k x 384 embedding table).

Fixed batch: **32 sequences x 512 tokens = 16,384 tokens/step**, 3,051 steps = 50M tokens.
AdamW (0.9, 0.95), lr 1e-3, 2% warm-up, cosine to 10%, wd 0.1, clip 1.0.
The LM head + cross-entropy is computed in checkpointed 4k-token chunks in *every* run, so the
50k-vocab logits never dominate memory and the comparison isolates the transformer blocks.

In [1]:
# --- setup: works on Google Colab (CUDA) and locally (Apple MPS / CUDA / CPU) ---
import os, sys
# MPS: make the allocator raise a clean OOM at the GPU's recommended working-set size
# instead of spilling into swap (must be set before torch is imported).
os.environ.setdefault("PYTORCH_MPS_HIGH_WATERMARK_RATIO", "1.0")
os.environ.setdefault("PYTORCH_MPS_LOW_WATERMARK_RATIO", "0.9")
if "google.colab" in sys.modules and not os.path.exists("revllm.py"):
    !git clone -q -b session13 https://github.com/chapanduk/ERAV5.git /content/ERAV5
    %cd /content/ERAV5/session13
    !pip -q install tiktoken pyarrow
if not os.path.exists("data/train.bin"):
    !python prepare_data.py
import json, torch, revllm
from revllm import ModelConfig, TrainConfig, train, pick_device
DEVICE = pick_device()
print("torch", torch.__version__, "| device", DEVICE,
      "|", torch.cuda.get_device_name() if DEVICE == "cuda" else "")
TRAIN_BIN, VAL_BIN = "data/train.bin", "data/val.bin"
GiB = 2**30

torch 2.15.0 | device mps | 


In [2]:
mc = ModelConfig(arch="standard")
tc = TrainConfig(name="01_baseline_standard_B32", batch_size=32)
res, model = train(mc, tc, TRAIN_BIN, VAL_BIN, DEVICE)

[01_baseline_standard_B32] arch=standard params=40.77M (non-emb 21.25M) device=mps B=32 T=512 tokens/step=16384 steps=3051


[01_baseline_standard_B32] step     1/3051 tok   0.02M loss 10.9247 gnorm 8.00   20.1k tok/s peak 7.00GiB


[01_baseline_standard_B32] step     2/3051 tok   0.03M loss 10.7087 gnorm 6.00   23.0k tok/s peak 7.31GiB


[01_baseline_standard_B32] step     3/3051 tok   0.05M loss 10.5053 gnorm 4.10   23.0k tok/s peak 7.31GiB


[01_baseline_standard_B32] step    50/3051 tok   0.82M loss 7.2485 gnorm 0.76   21.6k tok/s peak 7.31GiB


[01_baseline_standard_B32] step   100/3051 tok   1.64M loss 6.8037 gnorm 0.39   20.0k tok/s peak 7.31GiB


... (67 outputs left out)

[01_baseline_standard_B32] step  3050/3051 tok  49.97M loss 4.6174 gnorm 0.64   16.9k tok/s peak 7.31GiB


[01_baseline_standard_B32] step  3051/3051 tok  49.99M loss 4.6663 gnorm 0.59   16.9k tok/s peak 7.31GiB


[01_baseline_standard_B32]   val_loss 4.6310


[01_baseline_standard_B32] DONE final_train_loss=4.6568 final_val_loss=4.6310 tok/s=17012 peak=7.31GiB time=49.0min


In [3]:
print(json.dumps({k: res[k] for k in ["final_train_loss", "final_val_loss", "tokens_per_sec",
      "peak_mem_bytes", "driver_peak_mem_bytes", "train_time_s", "params_total", "params_non_embedding"]}, indent=1))
print(f"peak memory: {res['peak_mem_bytes']/GiB:.2f} GiB")

{
 "final_train_loss": 4.656822395324707,
 "final_val_loss": 4.631029844284058,
 "tokens_per_sec": 17012.41913358579,
 "peak_mem_bytes": 7845094656,
 "driver_peak_mem_bytes": 12369821696,
 "train_time_s": 2937.6492461233865,
 "params_total": 40766208,
 "params_non_embedding": 21252864
}
peak memory: 7.31 GiB
